# 3. Becoming the circuit: a hand-written RTL core

### What is inside an FPGA?

Not a processor. A sheet of:

- **LUTs** (look-up tables) - tiny configurable truth tables that become any logic gate;
- **flip-flops** - one-bit registers that hold state between clock edges;
- **DSP blocks** - hardened multiply-accumulate units (real multiplier silicon);
- **BRAM** - small on-chip memories.

You don't write a *program* that runs on it; you *describe a circuit* and the fabric
*becomes* that circuit. For our convolution that means: nine DSP multipliers wired into
one adder tree, all firing on the **same clock edge**, producing one output pixel per
cycle. No loop over the nine taps - they are nine wires.

The hand-written core is `rtl/conv3x3_core.sv` (with line buffers so a raster pixel
stream forms 3x3 windows on the fly). It is pipelined to meet timing and is bit-exact
with `conv_reference`.

### The catch: getting data in and out

A fast datapath is useless if you can't feed it. This first version is driven the
simplest possible way - the CPU writes each pixel to a hardware register over the AXI
bus (memory-mapped I/O, **MMIO**), one bus transaction per pixel, and reads each result
back the same way. That bus round-trip, not the arithmetic, is what we are about to
measure.

> **Board only.** The rungs below need the PYNQ board (`pynq` + the bitstream). On a
> laptop `available_backends()` omits them and the cell prints a skip message - that is
> expected; run this one on the board.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from fpga_conv import KERNELS, get_kernel, conv_reference, to_grayscale_u8, available_backends
from fpga_conv.bench import (
    Scoreboard, run_rung, time_backend, sample_gray, make_sample_image, roofline,
)

# Path to the combined bitstream (carries all four accelerators). Its .hwh must sit
# beside it. Relative to this notebook on the dev box; on the board point it at wherever
# you copied the build, e.g. '/home/xilinx/workshop/conv3x3.bit'.
BIT_PATH = os.path.abspath('../../rtl/build/conv3x3.bit')
HW_KW = {'bitfile': BIT_PATH}          # passed to the hardware rungs only

# The scoreboard persists across the whole series (notebooks/scoreboard.json), so the
# roofline grows as you climb. The hardware rungs are skipped off the board.
sb = Scoreboard()
print('backends available here:', available_backends())

In [ ]:
# Small image on purpose: this path streams one pixel per MMIO write, so wall-clock
# grows with pixel count - which is exactly the lesson.
if 'rtl' in available_backends():
    img = sample_gray(96)
    out, t = run_rung('rtl', img, 'edges', repeats=5, scoreboard=sb, **HW_KW)
else:
    print('rtl backend not available here (laptop) - run this cell on the board.')

The compute is fully parallel - nine multipliers every clock - yet the throughput is
*poor*. Why? Every pixel crosses the AXI bus twice (write in, read out) under CPU
control, and each crossing costs far more than the nine multiplies it feeds. The
datapath spends almost all its time idle, waiting for data. **The bottleneck moved from
compute to data movement.**

On the roofline this rung sits low not because it can't compute, but because we are
trickling data to it by hand.

In [ ]:
sb.roofline(); plt.show()

**Next:** writing every wire by hand is slow going. Let a tool compile C++ into the
circuit - High-Level Synthesis - and let the accelerator fetch its own data from DRAM.
-> `04_hls_naive.ipynb`.

*(Side quest: how fast can this core be clocked before it stops working? See the bonus
notebook `07_clock_ramp.ipynb`.)*